# 🟢 LEVEL 1 — BASICS (Foundation)
## 1️⃣ What is Spark?

Apache Spark is:

Distributed data processing engine

In-memory computing

Faster than Hadoop MapReduce

Used for Big Data, ETL, ML, Streaming

## Spark Components:

Spark Core

Spark SQL

Spark Streaming

MLlib

GraphX

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LearningSpark").getOrCreate()


In [5]:
spark

# 🟢 LEVEL 2 — DataFrame Basics

## 4️⃣ Create DataFrame

In [58]:
data = [
    (1, "Alice", 25, 5000),
    (2, "Bob", 30, 6000),
    (3, "Charlie", 35, 7000)
]

columns = ["id", "name", "age", "salary"]

df = spark.createDataFrame(data, columns)
df.show()


+---+-------+---+------+
| id|   name|age|salary|
+---+-------+---+------+
|  1|  Alice| 25|  5000|
|  2|    Bob| 30|  6000|
|  3|Charlie| 35|  7000|
+---+-------+---+------+



## 5️⃣ Basic Operations

In [12]:
# Select Columns
df.select("name", "salary").show()

+-------+------+
|   name|salary|
+-------+------+
|  Alice|  5000|
|    Bob|  6000|
|Charlie|  7000|
+-------+------+



In [10]:
# Filter
df.filter(df.salary > 5500).show()

+---+-------+---+------+
| id|   name|age|salary|
+---+-------+---+------+
|  2|    Bob| 30|  6000|
|  3|Charlie| 35|  7000|
+---+-------+---+------+



In [19]:
# Add Column

from pyspark.sql.functions import col

df.withColumn("salary_after_bonus", col("salary") + 1000).show()


+---+-------+---+------+------------------+
| id|   name|age|salary|salary_after_bonus|
+---+-------+---+------+------------------+
|  1|  Alice| 25|  5000|              6000|
|  2|    Bob| 30|  6000|              7000|
|  3|Charlie| 35|  7000|              8000|
+---+-------+---+------+------------------+



# 🟡 LEVEL 3 — Transformations & Actions

## Transformations (Lazy)

select()

filter()

withColumn()

groupBy()

## Actions (Execute)

show()

count()

collect()

In [21]:
df.filter(df.age > 25).count()

2

# 🟡 LEVEL 4 — GroupBy & Aggregation

In [59]:
from pyspark.sql.functions import avg,max
df.groupBy("age").agg(
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary")
).show()

+---+----------+----------+
|age|avg_salary|max_salary|
+---+----------+----------+
| 25|    5000.0|      5000|
| 30|    6000.0|      6000|
| 35|    7000.0|      7000|
+---+----------+----------+



# 🟡 LEVEL 5 — Joins

## Types of joins:
inner

left

right

outer

In [62]:
dept_data = [
    (1, "HR"),
    (2, "IT"),
    (3, "Finance")
]

df_dept = spark.createDataFrame(dept_data, ["id", "department"])

df.join(df_dept, on="id", how="inner").show()


+---+-------+---+------+----------+
| id|   name|age|salary|department|
+---+-------+---+------+----------+
|  1|  Alice| 25|  5000|        HR|
|  2|    Bob| 30|  6000|        IT|
|  3|Charlie| 35|  7000|   Finance|
+---+-------+---+------+----------+



# 🟠 LEVEL 6 — Working with Files

In [66]:
# Read CSV
df = spark.read.csv("Salaries.csv", header=True, inferSchema=True)

In [67]:
df.show()

+---+------------------+--------------------+---------+-----------+---------+--------+---------+----------------+----+-----+-------------+------+
| Id|      EmployeeName|            JobTitle|  BasePay|OvertimePay| OtherPay|Benefits| TotalPay|TotalPayBenefits|Year|Notes|       Agency|Status|
+---+------------------+--------------------+---------+-----------+---------+--------+---------+----------------+----+-----+-------------+------+
|  1|    NATHANIEL FORD|GENERAL MANAGER-M...|167411.18|        0.0|400184.25|    NULL|567595.43|       567595.43|2011| NULL|San Francisco|  NULL|
|  2|      GARY JIMENEZ|CAPTAIN III (POLI...|155966.02|  245131.88|137811.38|    NULL|538909.28|       538909.28|2011| NULL|San Francisco|  NULL|
|  3|    ALBERT PARDINI|CAPTAIN III (POLI...|212739.13|  106088.18|  16452.6|    NULL|335279.91|       335279.91|2011| NULL|San Francisco|  NULL|
|  4| CHRISTOPHER CHONG|WIRE ROPE CABLE M...|  77916.0|   56120.71| 198306.9|    NULL|332343.61|       332343.61|2011| NULL|

In [68]:
# Write CSV 
df.write.mode("overwrite").csv("output_folder", header=True)

# 🟠 LEVEL 7 — Spark SQL

In [74]:
df.createOrReplaceTempView("employees")

spark.sql("""
SELECT JobTitle, AVG(TotalPayBenefits)
FROM employees
GROUP BY JobTitle
""").show()


+--------------------+---------------------+
|            JobTitle|avg(TotalPayBenefits)|
+--------------------+---------------------+
|MANAGER, UNIFIED ...|            124392.29|
|COURT ALTERNATIVE...|   58886.517499999994|
|       CITY ATTORNEY|            208243.12|
|HEALTH PROGRAM CO...|    80969.15729166666|
|PERFORMANCE ANALY...|    94262.28166666666|
|HEAVY EQUIPMENT O...|              92872.0|
|          Dep Dir IV|   218886.99797297298|
|          Manager VI|    202949.6352525253|
|Communications Li...|   126014.70000000001|
|BATTALION CHIEF, ...|    216655.5368888889|
|ASSOCIATE PERFORM...|    56206.88909090909|
|SENIOR ESTATE INV...|             81456.45|
|              ROOFER|    68091.44888888889|
|        Undersheriff|            166720.53|
|        Dept Head II|   224066.75999999998|
|Sprv Electronic M...|          172242.2175|
|AUTOMOTIVE MECHAN...|   126170.64000000001|
|MANAGER VII - MUN...|           150641.082|
|STATION AGENT, MU...|    94206.51288461537|
|CONFIDENT

# 🔵 LEVEL 8 — Window Functions (Advanced)

In [78]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("JobTitle").orderBy("BasePay")

df.withColumn("rank", row_number().over(window_spec)).show()


+-----+----------------+-------------+--------+-----------+--------+--------+--------+----------------+----+-----+-------------+------+----+
|   Id|    EmployeeName|     JobTitle| BasePay|OvertimePay|OtherPay|Benefits|TotalPay|TotalPayBenefits|Year|Notes|       Agency|Status|rank|
+-----+----------------+-------------+--------+-----------+--------+--------+--------+----------------+----+-----+-------------+------+----+
|35638|   TSE SHIH LENG|ACCOUNT CLERK|   614.0|        0.0|     0.0|    NULL|   614.0|           614.0|2011| NULL|San Francisco|  NULL|   1|
|34975|        LIEN LUM|ACCOUNT CLERK|  1630.0|        0.0|     0.0|    NULL|  1630.0|          1630.0|2011| NULL|San Francisco|  NULL|   2|
|34976|      RACHEL LAU|ACCOUNT CLERK|  1630.0|        0.0|     0.0|    NULL|  1630.0|          1630.0|2011| NULL|San Francisco|  NULL|   3|
|32036|  BARBARA BLOUNT|ACCOUNT CLERK| 8681.58|      69.82|     0.0|    NULL|  8751.4|          8751.4|2011| NULL|San Francisco|  NULL|   4|
|31246|   ALI

# 🔵 LEVEL 9 — Performance Optimization

In [80]:
# 1️⃣ Partitioning
df.repartition(4)

DataFrame[Id: int, EmployeeName: string, JobTitle: string, BasePay: double, OvertimePay: double, OtherPay: double, Benefits: double, TotalPay: double, TotalPayBenefits: double, Year: int, Notes: string, Agency: string, Status: double]

In [83]:
# 2️⃣ Caching
df.cache()

26/02/13 17:00:49 WARN CacheManager: Asked to cache already cached data.


DataFrame[Id: int, EmployeeName: string, JobTitle: string, BasePay: double, OvertimePay: double, OtherPay: double, Benefits: double, TotalPay: double, TotalPayBenefits: double, Year: int, Notes: string, Agency: string, Status: double]

In [85]:
# 3️⃣ Explain Plan
df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [Id#1348, EmployeeName#1349, JobTitle#1350, BasePay#1351, OvertimePay#1352, OtherPay#1353, Benefits#1354, TotalPay#1355, TotalPayBenefits#1356, Year#1357, Notes#1358, Agency#1359, Status#1360]
      +- InMemoryRelation [Id#1348, EmployeeName#1349, JobTitle#1350, BasePay#1351, OvertimePay#1352, OtherPay#1353, Benefits#1354, TotalPay#1355, TotalPayBenefits#1356, Year#1357, Notes#1358, Agency#1359, Status#1360], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- FileScan csv [Id#1348,EmployeeName#1349,JobTitle#1350,BasePay#1351,OvertimePay#1352,OtherPay#1353,Benefits#1354,TotalPay#1355,TotalPayBenefits#1356,Year#1357,Notes#1358,Agency#1359,Status#1360] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/naseem.mohammad@e2erspl.COM/Github/celery_redis/Python_Libr..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Id:int,EmployeeName:string,Job

# 🔵 LEVEL 10 — Real ETL Project Example

## Problem:
You have sales data.
Find total revenue per product.

In [86]:
sales = [
    ("Laptop", 1000),
    ("Mobile", 500),
    ("Laptop", 1200),
    ("Mobile", 700)
]

df = spark.createDataFrame(sales, ["product", "price"])

from pyspark.sql.functions import sum

df.groupBy("product") \
  .agg(sum("price").alias("total_revenue")) \
  .show()


+-------+-------------+
|product|total_revenue|
+-------+-------------+
| Laptop|         2200|
| Mobile|         1200|
+-------+-------------+



# 🟣 LEVEL 11 — RDD (Low Level)

In [87]:
rdd = spark.sparkContext.parallelize([1,2,3,4,5])
rdd.map(lambda x: x*2).collect()


[2, 4, 6, 8, 10]

In [89]:
# 🟣 LEVEL 12 — Real Industry Topics

# To become job-ready, learn:

# Partition tuning

# Broadcast joins

# Shuffle operations

# Skew handling

# Spark UI

# Memory management

# Delta Lake

# Spark on Kubernetes